# Benchmark: OTIS Application 1 — Uvas Creek Chloride Transport

## Motivation

This notebook replicates the classic **Uvas Creek Chloride Transport** benchmark (Application 1) from the OTIS (One-Dimensional Transport with Inflow and Storage) model. The original field experiment and parameterization are detailed in Bencala and Walters (1983) and Runkel (1998, Figures 15–17).

We simulate the downstream transport of a conservative tracer (Chloride) injected into a stream, accounting for advection, dispersion, lateral inflow, and transient storage (dead zones). This serves as a rigorous validation test for the `RiverSoluteTransportDynamics` Landlab component.

## Physical Setup & Parameters

The 619-meter study reach is divided into 5 sub-reaches with distinct hydraulic and transport properties:

| Reach | Start [m] | End [m] | Disp. [m²/s] | Area [m²] | Storage Area [m²] | Exchange $\alpha$ [s⁻¹] | Lateral Inflow [m³/s/m] |
|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| 1 | 0 | 38 | 0.12 | 0.30 | 0.05 | 0.0 | 0.0 |
| 2 | 38 | 105 | 0.15 | 0.42 | 0.05 | 0.0 | 0.0 |
| 3 | 105 | 281 | 0.24 | 0.36 | 0.36 | 3.0e-5 | 4.545e-6 |
| 4 | 281 | 433 | 0.31 | 0.41 | 0.41 | 1.0e-5 | 1.974e-6 |
| 5 | 433 | 669 | 0.40 | 0.52 | 1.56 | 4.5e-5 | 2.151e-6 |

An injection of 11.4 mg/L of Chloride occurs between $t = 8.4$ and $t = 11.4$ hours. Background concentration is 3.7 mg/L.


## 1. Imports

In [ ]:
import sys

import matplotlib.pyplot as plt
import numpy as np

from landlab import RasterModelGrid

sys.path.insert(0, "/home/claude")
from landlab.components import RiverSoluteTransportDynamics

## 2. OTIS Reach Parameters & Global Forcing

We define the sub-reach geometries, dispersion coefficients, and transient storage exchange rates as outlined in the OTIS Uvas Creek application.

In [ ]:
# ======================================================================
# OTIS parameters
# ======================================================================
reach_starts = np.array([0, 38, 105, 281, 433])
reach_ends = np.array([38, 105, 281, 433, 669])

DISP = np.array([0.12, 0.15, 0.24, 0.31, 0.40])
AREA = np.array([0.30, 0.42, 0.36, 0.41, 0.52])
AREA2 = np.array([0.05, 0.05, 0.36, 0.41, 1.56])
ALPHA = np.array([0.0, 0.0, 3.0e-5, 1.0e-5, 4.5e-5])
QLATIN = np.array([0.0, 0.0, 4.545e-6, 1.974e-6, 2.151e-6])
CLATIN = np.array([3.7, 3.7, 3.7, 3.7, 3.7])

# Global forcing & timing parameters
Q0 = 0.0125  # Baseflow [m³/s]
C_BG = 3.7  # Background concentration [mg/L]
C_INJ = 11.4  # Injection concentration [mg/L]

INJ_START = 8.4  # Injection start time [hr]
INJ_END = 11.4  # Injection end time [hr]
T_START = 8.25  # Simulation start time [hr]
T_FINAL = 24.0  # Simulation end time [hr]

# Monitoring stations [m]
MONITOR = [38, 105, 281, 433, 619]

## 3. Grid Setup & Spatial Mapping

We construct a 3-row, 670-column grid to represent the 1D channel reach. We then map the 5 sets of reach parameters onto the per-column grid arrays and calculate the cumulative streamflow $Q(x)$ considering lateral inflows.

In [ ]:
# ======================================================================
# Build grid
# ======================================================================
dx = 1.0
n_cols = 670
n_rows = 3
grid = RasterModelGrid((n_rows, n_cols), xy_spacing=dx)


# Map reach parameters to per-column arrays
def rmap(col):
    for r in range(5):
        if reach_starts[r] <= col < reach_ends[r]:
            return r
    return 4


disp_1d = np.array([DISP[rmap(c)] for c in range(n_cols)])
area_1d = np.array([AREA[rmap(c)] for c in range(n_cols)])
area2_1d = np.array([AREA2[rmap(c)] for c in range(n_cols)])
alpha_1d = np.array([ALPHA[rmap(c)] for c in range(n_cols)])
qlatin_1d = np.array([QLATIN[rmap(c)] for c in range(n_cols)])
clatin_1d = np.array([CLATIN[rmap(c)] for c in range(n_cols)])

# Cumulative flow calculation
Q_1d = np.zeros(n_cols)
Q_1d[0] = Q0
for c in range(1, n_cols):
    Q_1d[c] = Q_1d[c - 1] + qlatin_1d[c] * dx

# Kinematic approximations
vel_1d = Q_1d / area_1d
h_1d = area_1d.copy()
hs_1d = area2_1d.copy()  # h_storage = AREA2 / width (width = 1m)


def tile(a):
    return np.tile(a, n_rows)

## 4. Initialization of Hydraulic Fields

Because we are only tracking solute transport on a prescribed steady-state flow field, we directly map the steady water depth, advection velocity, and lateral specific discharge to the Landlab grid rather than running a hydrodynamic solver.

In [ ]:
# ======================================================================
# Hydraulic fields
# ======================================================================
_ = grid.add_zeros("surface_water__depth", at="node")
grid.at_node["surface_water__depth"][:] = tile(h_1d)

_ = grid.add_zeros("surface_water__velocity", at="link")
_ = grid.add_zeros("advection__velocity", at="link")

# Map steady velocities to horizontal links
for lid in grid.horizontal_links:
    tc = grid.node_at_link_tail[lid] % n_cols
    hc = grid.node_at_link_head[lid] % n_cols
    v = 0.5 * (vel_1d[tc] + vel_1d[hc])
    grid.at_link["surface_water__velocity"][lid] = v
    grid.at_link["advection__velocity"][lid] = v

# Lateral inflow as a grid field
_ = grid.add_zeros("lateral__water_specific_discharge", at="node")
grid.at_node["lateral__water_specific_discharge"][:] = tile(qlatin_1d)

## 5. Component Setup & Initial Conditions

We instantiate the `RiverSoluteTransportDynamics` component. Notice that we enforce a zero transverse dispersion coefficient to ensure strict 1D behavior and provide the arrays for transient storage (`alpha_exchange`, `h_storage`).

In [ ]:
# ======================================================================
# Component Configuration
# ======================================================================
rstd = RiverSoluteTransportDynamics(
    grid,
    solutes=["chloride"],
    dispersion_mode="isotropic",
    dispersion_coefficient=tile(disp_1d),
    transverse_dispersion_coefficient=0.0,  # 1D-equivalent: no cross-channel diffusion
    alpha_exchange={"chloride": tile(alpha_1d)},
    h_storage={"chloride": tile(np.maximum(hs_1d, 0.001))},
    cs_background={"chloride": C_BG},
    outlet_boundary_condition="zero_gradient",
)

# Initial conditions
grid.at_node["surface_water__chloride__concentration"][:] = C_BG
grid.at_node["storage_zone__chloride__concentration"][:] = C_BG
grid.at_node["lateral__chloride__concentration"][:] = tile(clatin_1d)

## 6. CFL Stability Analysis

To avoid numerical instability, we must satisfy the Courant-Friedrichs-Lewy (CFL) conditions for both advection and dispersion. We calculate limits based on the maximum velocity and maximum dispersion, taking a conservative 90% of the minimum threshold.

$$ dt_{adv} = \frac{dx}{u_{max}} \quad \text{and} \quad dt_{diff} = \frac{dx^2}{2 D_{max}} $$

In [ ]:
# ======================================================================
# CFL time step
# ======================================================================
dt_adv = dx / vel_1d.max()
dt_diff = dx**2 / (2.0 * disp_1d.max())
dt = 0.9 * min(dt_adv, dt_diff)
dt = min(dt, 1.0)

print("CFL Analysis:")
print(f"  v_max = {vel_1d.max():.4f} m/s")
print(f"  D_max = {disp_1d.max():.2f} m²/s")
print(f"  Chosen dt = {dt:.3f} s")

## 7. Main Simulation Loop

We run the simulation from $t = 8.25$ hr to $t = 24.0$ hr. During the injection window (8.4–11.4 hr), the inlet boundary condition is switched to $C_{INJ}$.

> **Note on Sampling:** To keep memory usage manageable and mirror the frequency of real-world probes, concentration snapshots at the monitoring nodes are recorded every 180 seconds.

In [ ]:
# ======================================================================
# Time loop
# ======================================================================
left = grid.nodes_at_left_edge
core_row = 1
mon_nodes = [core_row * n_cols + m for m in MONITOR]

total_s = (T_FINAL - T_START) * 3600.0
n_steps = int(np.ceil(total_s / dt))

# Data sampling parameters
PSTEP = 180.0
rec_int = max(1, int(PSTEP / dt))
max_rec = n_steps // rec_int + 2

t_rec = np.zeros(max_rec)
c_rec = np.zeros((max_rec, len(MONITOR)))
ri = 0

print(f"Running Grid: {n_rows}x{n_cols}, dt={dt:.3f}s for {n_steps} steps...")

time_s = T_START * 3600.0
C_field = grid.at_node["surface_water__chloride__concentration"]

for step in range(n_steps):
    thr = time_s / 3600.0

    # Boundary Condition Toggle (Injection Window)
    bc = C_INJ if INJ_START <= thr < INJ_END else C_BG
    C_field[left] = bc

    # Advance Solver
    rstd.run_one_step(dt)

    # Re-apply BC (ghost cells)
    C_field[left] = bc
    time_s += dt

    # Log Data
    if step % rec_int == 0:
        t_rec[ri] = time_s / 3600.0
        for k, nid in enumerate(mon_nodes):
            c_rec[ri, k] = C_field[nid]
        ri += 1

    # Print Progress
    if step % 10000 == 0 and step > 0:
        print(
            f"  step {step:6d}/{n_steps}  t={thr:6.2f}hr  "
            f"C_105m={C_field[mon_nodes[1]]:.2f} mg/L  "
            f"C_433m={C_field[mon_nodes[3]]:.2f} mg/L"
        )

t_rec = t_rec[:ri]
c_rec = c_rec[:ri]
print(f"\nSimulation complete: {n_steps} steps, {ri} records generated.")

## 8. OTIS Benchmark Comparison & Plotting

We plot our results against the historical field observations digitised from Runkel (1998, Figure 17). The simulated breakthrough curves at 105 m and 433 m should tightly trace the observed OTIS data points.

In [ ]:
# ======================================================================
# Observed data (digitised from OTIS Figure 17)
# ======================================================================
obs_105_t = np.array(
    [
        8.051,
        8.45,
        8.724,
        8.798,
        9.134,
        9.222,
        9.256,
        9.371,
        9.471,
        9.5,
        9.6,
        9.65,
        9.8,
        10,
        10.729,
        10.94,
        11.065,
        11.102,
        11.401,
        11.7,
        12,
        12.05,
        12.2,
        12.285,
        12.36,
        12.472,
        12.534,
        12.6,
        12.8,
        13,
        13.5,
        14,
        15,
        16,
        17,
        18,
        19,
        20,
        21,
        22,
        23,
        23.01,
        23.05,
    ]
)
obs_105_c = np.array(
    [
        3.7,
        3.7,
        3.7,
        3.9,
        4.1,
        5.4,
        7.15,
        8.2,
        9.35,
        10.2,
        10.55,
        10.85,
        11.05,
        11.25,
        11.4,
        11.4,
        11.45,
        11.45,
        11.5,
        11.43,
        10.9,
        10.4,
        8.95,
        7.65,
        6.3,
        5.6,
        4.9,
        4.5,
        4,
        3.7,
        3.7,
        3.7,
        3.7,
        3.7,
        3.7,
        3.7,
        3.7,
        3.7,
        3.7,
        3.7,
        3.7,
        3.7,
        3.7,
    ]
)

obs_433_t = np.array(
    [
        7.702,
        7.852,
        9.508,
        9.835,
        10.504,
        10.938,
        11.377,
        11.5,
        11.7,
        11.9,
        12.05,
        12.2,
        12.3,
        12.55,
        12.67,
        12.79,
        13.16,
        13.5,
        14.03,
        14.352,
        14.53,
        14.85,
        15,
        15.2,
        15.35,
        15.55,
        15.8,
        16.326,
        16.795,
        17.505,
        17.989,
        18.375,
        18.636,
        19.371,
        19.5,
        19.633,
        20.355,
        20.654,
        22.496,
    ]
)
obs_433_c = np.array(
    [
        3.7,
        3.7,
        3.7,
        3.7,
        3.7,
        3.7,
        4.1,
        4.7,
        5.55,
        6.3,
        6.95,
        7.55,
        8,
        8.3,
        8.6,
        8.737,
        9.35,
        9.15,
        9.3,
        8.8,
        8.2,
        6.75,
        6.05,
        5.6,
        5.25,
        4.8,
        4.33,
        4.1,
        4.05,
        3.9,
        3.9,
        3.9,
        3.9,
        3.9,
        3.9,
        3.9,
        3.9,
        3.9,
        3.9,
    ]
)

# ======================================================================
# Plot
# ======================================================================
fig, ax = plt.subplots(figsize=(11, 6))
colors = ["#4e79a7", "#f28e2b", "#59a14f", "#e15759", "#b07aa1"]

# Plot RSTD simulated time series
for k, d in enumerate(MONITOR):
    ax.plot(t_rec, c_rec[:, k], color=colors[k], lw=2.2, label=f"RSTD {d} m")

# Plot Observed OTIS data
ax.plot(
    obs_105_t,
    obs_105_c,
    "s",
    color=colors[1],
    ms=7,
    mfc="none",
    mew=1.8,
    label="Obs 105 m",
)
ax.plot(
    obs_433_t,
    obs_433_c,
    "D",
    color=colors[3],
    ms=7,
    mfc="none",
    mew=1.8,
    label="Obs 433 m",
)

ax.set_xlabel("Time [hour]", fontsize=12)
ax.set_ylabel("Chloride Concentration [mg/L]", fontsize=12)
ax.set_title(
    "Uvas Creek Chloride Transport - RSTD vs OTIS Application 1",
    fontsize=14,
    fontweight="bold",
)

ax.set_xlim(5, 25)
ax.set_ylim(2.5, 13)
ax.legend(loc="upper right", fontsize=10, ncol=2)
ax.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

# ======================================================================
# Summary
# ======================================================================
print("\n" + "=" * 65)
print("BENCHMARK SUMMARY")
print("=" * 65)
for k, d in enumerate(MONITOR):
    pk = np.max(c_rec[:, k])
    pt = t_rec[np.argmax(c_rec[:, k])]
    print(f"  {d:4d} m:  Peak = {pk:6.2f} mg/L at t = {pt:5.2f} hr")

print("\nOTIS Reference (Runkel 1998, Figure 17):")
print("  105 m: Peak ~ 10.8 mg/L at ~ 11.3 hr")
print("  433 m: Peak ~  7.2 mg/L at ~ 13.5 hr")
print("=" * 65)